In [7]:
from astropy.io import ascii
from astropy.table import Table, join
import numpy as np

In [18]:
meta = Table(ascii.read('cube_metadata.csv'))
clusterz = Table(ascii.read('clusters.csv'))
master = join(meta,clusterz)
master['z'][master['z']==0] = np.nan
for m in master:
    m['TIME'] = m['TIME'][:8]

In [ ]:
def astropy_to_latex(table, output_file=None, caption="", label="", col_format=None):
    """
    Convert an Astropy Table to a booktabs-style LaTeX table.

    Parameters
    ----------
    table : astropy.table.Table
    output_file : str, optional
        If given, write output to this .tex file.
    caption : str, optional
    label : str, optional
    col_format : str, optional
        LaTeX column format string, e.g. "lcc". Defaults to all centred.
    """
    headers = table.colnames
    units = [str(table[col].unit) if table[col].unit else None for col in headers]
    has_units = any(u and u != "None" for u in units)

    if col_format is None:
        col_format = "l" + "c" * (len(headers) - 1)  # first col left, rest centred

    def escape(val):
        # Replace missing values
        if val is None:
            return r"\ldots"
        try:
            if np.isnan(float(val)):
                return r"\ldots"
        except (ValueError, TypeError):
            pass
        if str(val).strip() in ("N/A", "nan", "None", ""):
            return r"\ldots"

        s = str(val)
        s = s.replace("_", r"\_")
        s = s.replace("%", r"\%")
        s = s.replace("&", r"\&")
        s = s.replace("#", r"\#")
        s = s.replace("$", r"\$")
        s = s.replace("~", r"\textasciitilde{}")
        s = s.replace("^", r"\^{}")
        return s

    lines = []
    
    lines.append(r"\begin{sidewaystable}")
    lines.append(r"    \centering")
    if caption:
        lines.append(f"    \\caption{{{caption}}}")
    if label:
        lines.append(f"    \\label{{{label}}}")
    lines.append(f"    \\begin{{tabular}}{{{col_format}}}")

    lines.append(r"        \toprule")

    # Header row
    lines.append("        " + " & ".join(f"\\textbf{{{escape(h)}}}" for h in headers) + r" \\")

    # Optional units row
    if has_units:
        unit_cells = [f"({escape(u)})" if u and u != "None" else "" for u in units]
        lines.append("        " + " & ".join(unit_cells) + r" \\")

    lines.append(r"        \midrule")

    # Data rows
    for row in table:
        cells = [escape(row[col]) for col in headers]
        lines.append("        " + " & ".join(cells) + r" \\")

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{sidewaystable}")

    result = "\n".join(lines)

    if output_file:
        with open(output_file, "w") as f:
            f.write(result)
        print(f"Written to {output_file}")
    else:
        print(result)

In [21]:
astropy_to_latex(master)

\begin{table}
    \centering
    \begin{tabular}{lcccccccc}
        \toprule
        \textbf{CUBEKEY} & \textbf{DIR} & \textbf{DATE} & \textbf{TIME} & \textbf{OBJECT} & \textbf{FWHM\_XY\_AVG} & \textbf{FWHM\_MAX} & \textbf{FWHM\_MIN} & \textbf{z} \\
        \midrule
        a141 & cubes & 2023-08-21 & 20:53:46 & Abell141 & \ldots & \ldots & \ldots & 0.23 \\
        a209 & cubes & 2019-11-25 & 12:41:47 & A209 & 0.903 & 1.146 & 0.781 & 0.2048 \\
        a2697 & cubes & 2023-09-19 & 15:47:51 & Abell2697 & \ldots & \ldots & \ldots & 0.234 \\
        a2811 & cubes & 2023-08-16 & 01:23:16 & Abell2811 & \ldots & \ldots & \ldots & 0.1075 \\
        a2813 & cubes & 2022-09-15 & 02:49:34 & A2813 & \ldots & \ldots & \ldots & 0.2924 \\
        a3017 & cubes & 2019-11-20 & 12:11:41 & A3017 & 0.754 & 0.939 & 0.651 & 0.2195 \\
        a3186 & cubes\_new & 2022-09-10 & 20:36:16 & A3186 & \ldots & \ldots & \ldots & 0.127 \\
        a3230 & cubes & 2023-10-09 & 00:21:16 & A3230 & \ldots & \ldots & \ldot